# R5 Seed 123: Ablation - Shuffled-Rationale & Plain-Continuation

**Purpose:** Run R5 ablations for a single seed.

**Prerequisite:** R4 must be completed first - this notebook loads Stage 1 adapter for this seed.

Two ablations for this seed:
1. **Shuffled-Rationale:** Stage 2 with randomly reassigned rationales
2. **Plain-Continuation:** Stage 2 on same texts in classification format (no CoT)

**Fix in this version:** Plain-Continuation includes Normal-class samples in Stage 2 data
to prevent Normal-class collapse.

**Output:** `results_ablations_seed123.json`


## Step 1: Install Dependencies

In [ ]:
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.0
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy tqdm openpyxl
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes
!pip install -q peft

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
warnings.filterwarnings('ignore')
print('Dependencies installed')

## Step 2: Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, sys, os, json, time
from pathlib import Path

# === CONFIGURE THIS ===
DRIVE_BASE = Path('/content/drive/MyDrive/ViTHSD')
# ======================

SEED = 123

WORK_DIR = Path('/content/vithsd')
WORK_DIR.mkdir(exist_ok=True)

for f in ['config.py', 'data_preparation.py', 'evaluation.py', 'models.py']:
    src = DRIVE_BASE / 'src' / f
    dst = WORK_DIR / f
    if src.exists():
        shutil.copy2(src, dst)
        print(f'  Copied {f}')
    else:
        raise FileNotFoundError(f'Missing: {src}')

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))

ADAPTER_DIR = DRIVE_BASE / 'adapters'
OUTPUT_DIR = DRIVE_BASE / 'outputs' / 'R5_ablations'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import config
config.DATA_DIR = DRIVE_BASE / 'data'
config.OUTPUT_DIR = OUTPUT_DIR

adapter_path = ADAPTER_DIR / f'stage1_seed{SEED}' / 'lora_adapter' / 'adapter_config.json'
assert adapter_path.exists(), f'Missing Stage 1 adapter for seed {SEED}. Run R4 first!'
print(f'Stage 1 adapter found for seed {SEED}')

import torch
print(f'\nGPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB)')
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
BATCH_SIZE = 8 if gpu_mem >= 35 else (4 if gpu_mem >= 14 else 2)


## Step 3: Load Data

In [ ]:
from config import FINAL_LABELS
from data_preparation import load_dataset_A
from evaluation import Evaluator
from models import create_model

train_texts, train_labels = load_dataset_A('train', data_dir=config.DATA_DIR)
test_texts, test_labels = load_dataset_A('test', data_dir=config.DATA_DIR)

rationale_path = DRIVE_BASE / 'data' / 'dataset_rationale.json'
with open(rationale_path, 'r', encoding='utf-8') as f:
    rationale_data = json.load(f)

r_train_texts, r_train_implied, r_train_rationale, r_train_labels = [], [], [], []
for r in rationale_data:
    if str(r.get('dataset', '')).lower().strip() != 'train':
        continue
    content = (r.get('content') or '').strip()
    implied = (r.get('implied_statement') or '').strip()
    if not content or not implied:
        continue
    r_train_texts.append(content)
    r_train_implied.append(implied)
    r_train_rationale.append(r.get('rationale', []))
    r_train_labels.append(r.get('labels', []))

# Count Normal samples in rationale data
normal_count = sum(1 for l in r_train_labels if l == ['normal'])
print(f'Train: {len(train_texts)} | Test: {len(test_texts)}')
print(f'Rationale samples: {len(r_train_texts)} ({normal_count} normal)')

## Step 4: Helper Functions

In [ ]:
import random, gc
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ensure_trainable(model, context):
    trainable_params = sum(param.numel() for param in model.model.parameters() if param.requires_grad)
    if trainable_params == 0:
        raise RuntimeError(
            f'No trainable parameters available during {context}. '
            'Sync the updated models.py to Drive, restart the runtime, and reload the adapter.'
        )
    print(f'  Trainable parameters ({context}): {trainable_params:,}')
    return trainable_params


def stage2_train(model, training_texts, num_epochs=1, learning_rate=5e-5):
    """Generic Stage 2 training on pre-formatted texts."""
    class TextDataset(Dataset):
        def __init__(self, texts, tokenizer, max_length):
            self.encodings = tokenizer(
                texts, truncation=True, padding=True,
                max_length=max_length, return_tensors='pt'
            )
        def __len__(self):
            return len(self.encodings['input_ids'])
        def __getitem__(self, idx):
            return {
                'input_ids': self.encodings['input_ids'][idx],
                'attention_mask': self.encodings['attention_mask'][idx]
            }

    dataset = TextDataset(training_texts, model.tokenizer, model.max_length)
    dataloader = DataLoader(dataset, batch_size=model.batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.model.parameters(), lr=learning_rate)

    ensure_trainable(model, 'R5 Stage 2 warmup')
    model.model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        progress = tqdm(dataloader, desc=f'  Stage2 Ep {epoch+1}/{num_epochs}')
        for batch in progress:
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            optimizer.zero_grad()
            outputs = model.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
            )
            loss = outputs.loss
            if not loss.requires_grad:
                raise RuntimeError(
                    'Stage 2 loss is detached from the computation graph. '
                    'Reload the adapter with the fixed source file before continuing.'
                )
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
            optimizer.step()
            progress.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f'  Epoch {epoch+1}: avg_loss = {total_loss / len(dataloader):.4f}')


def cleanup_model(model):
    if hasattr(model, 'model') and model.model is not None:
        del model.model
    if hasattr(model, 'tokenizer') and model.tokenizer is not None:
        del model.tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()


def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


print('Helpers defined')

## Step 5: Prepare Ablation Data

In [ ]:
def prepare_shuffled_data(model, seed, r_texts, r_implied, r_rationale, r_labels):
    """Create shuffled-rationale Stage 2 training data."""
    set_seed(seed)
    n = len(r_texts)
    shuffled_indices = list(range(n))
    random.shuffle(shuffled_indices)
    # Ensure derangement (no self-mapping)
    for i in range(n):
        if shuffled_indices[i] == i:
            j = (i + 1) % n
            shuffled_indices[i], shuffled_indices[j] = shuffled_indices[j], shuffled_indices[i]

    texts = []
    for i in range(n):
        input_text = model._format_input_cot(r_texts[i])
        output_text = model._format_output_cot(
            r_labels[i],                             # Original labels
            r_rationale[shuffled_indices[i]],         # Shuffled rationale
            r_implied[shuffled_indices[i]]            # Shuffled implied
        )
        texts.append(input_text + output_text)
    return texts


def prepare_plain_data(model, r_texts, r_labels, train_texts_all, train_labels_all):
    """
    Create plain-continuation Stage 2 training data.
    FIX: Include Normal samples from the full training set to prevent Normal collapse.
    """
    texts = []
    # Original 1,221 rationale samples in classification format
    for i in range(len(r_texts)):
        input_text = model._format_input_standard(r_texts[i])
        output_text = model._format_output_standard(r_labels[i])
        texts.append(input_text + output_text)

    # Add Normal samples from full training set
    # (rationale data is biased toward hate — mostly non-normal samples)
    normal_in_rationale = sum(1 for l in r_labels if l == ['normal'])
    normal_target = max(200, len(r_texts) // 4)  # At least 200 or 25% of data
    normal_needed = max(0, normal_target - normal_in_rationale)

    if normal_needed > 0:
        normal_indices = [i for i, l in enumerate(train_labels_all) if l == ['normal']]
        sampled = random.sample(normal_indices, min(normal_needed, len(normal_indices)))
        for idx in sampled:
            input_text = model._format_input_standard(train_texts_all[idx])
            output_text = model._format_output_standard(train_labels_all[idx])
            texts.append(input_text + output_text)
        print(f'  Added {len(sampled)} Normal samples to Plain-Continuation')

    return texts


print('Data preparation functions defined')

## Step 6: Run Configuration (Single Seed)


In [ ]:
ABLATIONS = ['shuffled', 'plain']

all_results = {}
all_predictions = {}

print(f'Configured single-seed run: seed={SEED}')
print(f'Ablations: {ABLATIONS} ({len(ABLATIONS)} runs)')


## Step 7: Run Ablations for This Seed


In [ ]:
evaluator = Evaluator()
s1_adapter_dir = str(ADAPTER_DIR / f'stage1_seed{SEED}')

for run_idx, ablation in enumerate(ABLATIONS, start=1):
    key = f'{ablation}_{SEED}'
    print(f'\n{"="*70}')
    print(f'RUN {run_idx}/{len(ABLATIONS)}: {ablation.upper()} seed={SEED}')
    print(f'{"="*70}')
    t0 = time.time()

    model = create_model(
        'qwen', dataset_type='A',
        batch_size=BATCH_SIZE, num_epochs=2,
        learning_rate=2e-4, lora_r=8, lora_alpha=16
    )
    model.load_adapter(s1_adapter_dir)

    set_seed(SEED)
    if ablation == 'shuffled':
        s2_texts = prepare_shuffled_data(
            model, SEED, r_train_texts, r_train_implied,
            r_train_rationale, r_train_labels
        )
    else:
        s2_texts = prepare_plain_data(
            model, r_train_texts, r_train_labels,
            train_texts, train_labels
        )

    print(f'  Stage 2 data: {len(s2_texts)} samples')
    set_seed(SEED)
    stage2_train(model, s2_texts, num_epochs=1, learning_rate=5e-5)

    print('  Predicting (free-text)...')
    pred, raw = model.predict(test_texts)
    res = evaluator.evaluate(test_labels, pred, f'{ablation}_seed{SEED}', 'ablation')

    print('  Predicting (constrained)...')
    pred_cd, scores_cd = model.predict_constrained(test_texts)
    res_cd = evaluator.evaluate(test_labels, pred_cd, f'{ablation}_seed{SEED}_constrained', 'ablation')

    elapsed = (time.time() - t0) / 60
    print(f'  DONE in {elapsed:.1f} min')
    print(f'    FreeText:    F1-Micro={res["f1_micro"]:.4f}')
    print(f'    Constrained: F1-Micro={res_cd["f1_micro"]:.4f}')

    all_results[key] = convert_to_serializable(res)
    all_results[f'{key}_constrained'] = convert_to_serializable(res_cd)
    all_predictions[key] = pred

    cleanup_model(model)

    partial_path = OUTPUT_DIR / f'results_ablations_seed{SEED}_partial.json'
    with open(partial_path, 'w') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    print('  Partial results saved')

print(f'\n{"="*70}')
print(f'ALL ABLATION RUNS COMPLETE FOR SEED {SEED}')
print(f'{"="*70}')


## Step 8: Summary & Save (Single Seed)


In [ ]:
print(f'{"SINGLE-SEED ABLATION SUMMARY":^70}')
print(f'{"="*70}')
print(f'Seed: {SEED}')
print(f'{"":>30} {"F1-Micro":>12} {"F1-Macro":>12}')
print(f'{"-"*55}')

for key in sorted(all_results.keys()):
    r = all_results[key]
    if 'f1_micro' in r:
        print(f'{key:>30} {r["f1_micro"]*100:10.2f}% {r["f1_macro"]*100:10.2f}%')

output = {
    'results': all_results,
    'predictions': convert_to_serializable(all_predictions),
    'seeds': [SEED],
    'note': 'Plain-Continuation includes Normal samples in Stage 2 to fix Normal collapse'
}

final_path = OUTPUT_DIR / f'results_ablations_seed{SEED}.json'
with open(final_path, 'w') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f'\nSaved: {final_path}')

p = OUTPUT_DIR / f'results_ablations_seed{SEED}_partial.json'
if p.exists():
    p.unlink()

print('DONE!')
